In [0]:
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, to_date, count, sum, avg, max, min, round

In [0]:
# Get the first day of the month two months ago
two_months_ago_start = date.today().replace(day=1) - relativedelta(months=2)

In [0]:
# Load cleansed yellow taxi trip data from the Silver layer
# and filter to only include trips with a pickup datetime
# later than the start date from two months ago
df_trips = spark.read.table("nyctaxi.02_silver.yellow_trips_cleansed").filter(f"tpep_pickup_datetime > '{two_months_ago_start}'")

# Load taxi zone lookup data from the Silver layer
df_zones = spark.read.table("nyctaxi.02_silver.taxi_zone_lookup")

In [0]:

df_daily_summary = df_trips.groupBy(df_trips.tpep_pickup_datetime.cast("date").alias("pickup_date")).agg(
        count("*").alias("total_trips"),
        round(avg("passenger_count"), 1).alias("average_passengers"),
        round(avg("trip_distance"), 1).alias("average_distance"),
        round(avg("fare_amount"), 2).alias("average_fare_per_trip"),
        max("fare_amount").alias("max_fare"),
        min("fare_amount").alias("min_fare"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )

In [0]:
# Write the daily summary to a Unity Catalog managed Delta table in the gold schema
df_daily_summary.write.mode("append").saveAsTable("nyctaxi.03_gold.daily_trip_summary")